# Notebook 11: MILK10k hidden benchmark submission

Scores the final v2 model on the 479 MILK10k benchmark lesions (ground truth is hidden; the ISIC server scores it).

1. Embed every benchmark image with all 15 v2 encoders (3 seeds x 5 folds, same preprocessing and TTA as notebook 07).
2. Refit the `late_drop` fusion head for each seed and fold on the stored training embeddings (the heads were never saved), exactly as notebook 08 did.
3. Average the 15 fused predictions per benchmark lesion.
4. The ISIC metric binarises every class at 0.5, so per-class thresholds are tuned on the out-of-fold training predictions and folded into the probabilities with a monotone map. The honest OOF estimate of this step comes from nested cross-validation.
5. Write the submission CSV (and a second one for the no-dermoscopy setting).

**Inputs:** MILK10_DATASET, the MILK10k benchmark data (test images + test metadata CSV) uploaded as a Kaggle dataset, and the three v2 training notebooks (seed 42, 43, 44). GPU T4, Internet on.

In [ ]:
import os, re, time, json, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from multiprocessing import Pool
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
import torch, torch.nn as nn, torch.nn.functional as F

INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input"); WORK = os.environ.get("KAGGLE_WORK", "/kaggle/working")
OUT = f"{WORK}/stage11"; CACHE = f"{WORK}/cache_test"
for d in [OUT, CACHE, f"{OUT}/test_emb"]: os.makedirs(d, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", device)

SEED = 42                       # split seed: inner validation split + fusion head seeds (same as notebook 08)
SUB_ID_COL = "lesion_id"        # the ISIC validator requires "lesion_id" (or "image")
MOD_CFG = {"D": dict(size=384, backbone="facebook/convnextv2-tiny-22k-384"),
           "C": dict(size=224, backbone="facebook/convnextv2-tiny-22k-224")}
COLOR_CONSTANCY = True          # must match notebook 07
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
MODS = ["C", "D", "M"]; PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]

# ---- v2 runs: every folder holding net_D_fold0.pt and emb_D_fold0.npy ----
RUNS = {}
for root, dirs, files in os.walk(INPUT, followlinks=True):
    if "net_D_fold0.pt" in files and "emb_D_fold0.npy" in files and os.path.basename(root).startswith("stage2v2_"):
        RUNS.setdefault(os.path.basename(root).replace("stage2v2_", "v2_"), root)
RUNS = dict(sorted(RUNS.items()))
need = [f"{k}_{m}_fold{f}.{e}" for k, e in [("net", "pt"), ("emb", "npy")] for m in "CD" for f in range(5)]
for r, d in RUNS.items():
    miss = [q for q in need if not os.path.exists(f"{d}/{q}")]
    assert not miss, f"{r} is missing {miss[:4]}: attach the committed v2 notebook output"
assert RUNS, "attach the v2 training notebooks (stage2v2_s42 / s43 / s44)"
print("runs:", RUNS)

# ---- benchmark metadata CSV (MILK10k_Test_Metadata.csv) ----
# 1) look in the attached inputs for a CSV whose lesions are NOT training lesions (works whatever the file is called)
# 2) if none is attached, download the official benchmark files from the ISIC archive (needs Internet on)
BENCH_URL = "https://isic-archive.s3.amazonaws.com/challenges/milk10k"
BENCH_DIR = f"{WORK}/milk10k_benchmark"
train_ids = set(pd.read_csv(f"{list(RUNS.values())[0]}/milk10k_lesions_folds.csv", usecols=["lesion_id"])["lesion_id"])

def is_benchmark_csv(path):
    try: h = pd.read_csv(path, nrows=5)
    except Exception: return False
    if not {"lesion_id", "isic_id", "image_type"} <= set(h.columns): return False
    ids = pd.read_csv(path, usecols=["lesion_id"])["lesion_id"]
    return ids.nunique() < 2000 and not ids.isin(train_ids).any()

TEST_META = None
for top in [INPUT, BENCH_DIR]:
    for root, dirs, files in os.walk(top, followlinks=True):
        for q in sorted(files):
            if q.lower().endswith(".csv") and is_benchmark_csv(os.path.join(root, q)):
                TEST_META = os.path.join(root, q); break
        if TEST_META: break
    if TEST_META: break

if TEST_META is None:
    import urllib.request, zipfile
    os.makedirs(BENCH_DIR, exist_ok=True)
    print("benchmark not attached: downloading it from the ISIC archive ...")
    try:
        for name in ["MILK10k_Test_Metadata.csv", "MILK10k_Test_Input.zip"]:
            dst = f"{BENCH_DIR}/{name}"
            if not os.path.exists(dst): urllib.request.urlretrieve(f"{BENCH_URL}/{name}", dst)
            print("  got", name, f"{os.path.getsize(dst) / 1e6:.1f} MB")
        with zipfile.ZipFile(f"{BENCH_DIR}/MILK10k_Test_Input.zip") as z: z.extractall(BENCH_DIR)
    except Exception as e:
        raise RuntimeError(f"Download failed ({e}). Turn Internet on (Settings > Internet), or download "
                           "MILK10k_Test_Metadata.csv and MILK10k_Test_Input.zip from https://challenge.isic-archive.com/data/#milk10k, "
                           "upload both as a Kaggle dataset and attach it with Add Input.")
    TEST_META = f"{BENCH_DIR}/MILK10k_Test_Metadata.csv"
    assert is_benchmark_csv(TEST_META), "downloaded CSV does not look like the benchmark metadata"
IMG_ROOTS = [INPUT, BENCH_DIR]            # where the benchmark images are searched (attached dataset or download)
print("benchmark metadata:", TEST_META)

In [ ]:
# ---- training lesion table (same folds for every run) ----
les = pd.read_csv(f"{list(RUNS.values())[0]}/milk10k_lesions_folds.csv")
for r, d in RUNS.items():
    o = pd.read_csv(f"{d}/milk10k_lesions_folds.csv")
    assert (o["lesion_id"].values == les["lesion_id"].values).all() and (o["fold"].values == les["fold"].values).all(), r
y_all = les["y"].values.astype(int); fold = les["fold"].values; N_TR = len(les)
MONET = [c[:-2] for c in les.columns if c.startswith("MONET_") and c.endswith("_C")]

# ---- benchmark lesion table, built the same way as the training table in notebook 07 ----
tm = pd.read_csv(TEST_META)
print("benchmark columns:", list(tm.columns))
if "site" not in tm.columns:
    alt = [c for c in tm.columns if "site" in c.lower()]
    tm["site"] = tm[alt[0]] if alt else np.nan
tm["mod"] = tm["image_type"].astype(str).str.lower().map(lambda s: "D" if "dermoscop" in s else ("C" if "clinical" in s else None))
assert tm["mod"].notna().all(), tm["image_type"].unique()
MON_T = [c for c in MONET if c in tm.columns]
USE_MONET = len(MON_T) == len(MONET) and tm[MON_T].notna().all().all()
w = tm.pivot(index="lesion_id", columns="mod", values=["isic_id"] + MON_T)
w.columns = [f"{a}_{b}" for a, b in w.columns]
test = tm[tm["mod"] == "D"][["lesion_id", "age_approx", "sex", "skin_tone_class", "site"]].merge(w.reset_index(), on="lesion_id")
test["skin_tone_class"] = test["skin_tone_class"].fillna(les["skin_tone_class"].median())
for c in MON_T:
    for m in "CD": test[f"{c}_{m}"] = test[f"{c}_{m}"].astype(float)
N_TE = len(test)
print(f"benchmark lesions: {N_TE} (expected 479) | MONET scores available: {USE_MONET}")
if not USE_MONET:
    print("MONET missing in the benchmark CSV, so this notebook fits the fusion heads WITHOUT MONET tokens "
          "(the image-only variant from notebook 10). Report that in the paper.")
test["tone_group"] = pd.cut(test["skin_tone_class"], [-1, 2, 3, 5], labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str)
display(pd.DataFrame({"benchmark": test["tone_group"].value_counts(normalize=True).round(3),
                      "training": les["tone_group"].value_counts(normalize=True).round(3)}))

In [ ]:
# ---- locate and cache the benchmark images (Shades of Gray + short side = 1.15 x input size, as in notebook 07) ----
want = set(test["isic_id_C"]) | set(test["isic_id_D"])
IMG = {}
for top in IMG_ROOTS:
  for root, dirs, files in os.walk(top, followlinks=True):
    for q in files:
        stem = os.path.splitext(q)[0]
        if stem in want and q.lower().endswith((".jpg", ".jpeg", ".png")): IMG.setdefault(stem, os.path.join(root, q))
missing = want - set(IMG); assert not missing, f"{len(missing)} benchmark images not found, e.g. {list(missing)[:3]}"

def shades_of_gray(im, p=6):
    a = np.asarray(im).astype(np.float32)
    illum = np.power(np.mean(np.power(a, p), axis=(0, 1)), 1 / p)
    illum = illum / (np.sqrt((illum ** 2).sum()) + 1e-6)
    a = a * (1.0 / (illum * np.sqrt(3) + 1e-6))
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

def cache_one(args):
    src, dst, short = args
    if os.path.exists(dst): return
    im = Image.open(src).convert("RGB"); w_, h_ = im.size; s = short / min(w_, h_)
    im = im.resize((max(short, round(w_ * s)), max(short, round(h_ * s))), Image.BICUBIC)
    if COLOR_CONSTANCY: im = shades_of_gray(im)
    im.save(dst, quality=95)

def cache_paths(ids, mod, tag):
    size = MOD_CFG[mod]["size"]; d = f"{CACHE}/{tag}_{mod}_{size}{'_cc' if COLOR_CONSTANCY else ''}"; os.makedirs(d, exist_ok=True)
    jobs = [(IMG[i], f"{d}/{i}.jpg", int(size * 1.15)) for i in ids]
    with Pool(4) as pool: pool.map(cache_one, jobs, chunksize=16)
    return [j[1] for j in jobs]

t0 = time.time()
for mod in ["C", "D"]: test[f"path_{mod}"] = cache_paths(test[f"isic_id_{mod}"], mod, "bench")
print(f"benchmark cache ready in {time.time() - t0:.0f}s")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from transformers import AutoConfig, AutoModel
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class LesionImages(Dataset):
    def __init__(self, paths, mod):
        s = MOD_CFG[mod]["size"]; self.p = list(paths)
        self.tf = T.Compose([T.Resize((s, s)), T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB"))

class Net(nn.Module):
    """Same module tree as notebook 07, built from the config only (the fine-tuned weights come from net_*.pt)."""
    def __init__(self, backbone):
        super().__init__()
        self.backbone = AutoModel.from_config(AutoConfig.from_pretrained(backbone))
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.config.hidden_sizes[-1], K))
    def embed(self, x): return self.backbone(pixel_values=x).pooler_output
    def forward(self, x): return self.head(self.embed(x))

def tta_views(x, mod):
    if mod == "D": return [torch.rot90(v, k, dims=(2, 3)) for v in (x, torch.flip(x, dims=[3])) for k in range(4)]
    return [x, torch.flip(x, dims=[3])]

@torch.no_grad()
def embed_images(model, paths, mod):
    dl = DataLoader(LesionImages(paths, mod), batch_size=32, shuffle=False, num_workers=2)
    model.eval(); probs, embs = [], []
    for x in dl:
        x = x.to(device).to(memory_format=torch.channels_last)
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            e = torch.stack([model.embed(v) for v in tta_views(x, mod)]).mean(0)
            p = model.head(e).float().softmax(1)
        probs.append(p.cpu().numpy()); embs.append(e.float().cpu().numpy())
    return np.concatenate(probs), np.concatenate(embs)

def load_net(run_dir, mod, f):
    net = Net(MOD_CFG[mod]["backbone"])
    net.load_state_dict(torch.load(f"{run_dir}/net_{mod}_fold{f}.pt", map_location="cpu"))   # strict: keys must match exactly
    return net.to(device).to(memory_format=torch.channels_last).eval()

In [ ]:
# ---- preprocessing check: re-embed 24 training lesions from raw images and compare with the stored embeddings ----
# If this cosine is not ~1, the benchmark images are being preprocessed differently from training. Stop and fix before going on.
r0, d0 = next(iter(RUNS.items()))
chk = les[les["fold"] == 0].head(24)
need_ids = set(chk["isic_id_C"]) | set(chk["isic_id_D"])
for root, dirs, files in os.walk(INPUT, followlinks=True):
    for q in files:
        if os.path.splitext(q)[0] in need_ids: IMG.setdefault(os.path.splitext(q)[0], os.path.join(root, q))
CHECK = {}
for mod in ["C", "D"]:
    net = load_net(d0, mod, 0)
    _, e_new = embed_images(net, cache_paths(chk[f"isic_id_{mod}"], mod, "check"), mod)
    e_old = np.load(f"{d0}/emb_{mod}_fold0.npy").astype(np.float32)[chk.index.values]
    cos = (e_new * e_old).sum(1) / (np.linalg.norm(e_new, axis=1) * np.linalg.norm(e_old, axis=1))
    CHECK[mod] = cos; print(f"{mod}: cosine new vs stored  min {cos.min():.4f}  mean {cos.mean():.4f}")
    del net
assert min(c.min() for c in CHECK.values()) > 0.98, "preprocessing mismatch with notebook 07"

In [ ]:
# ---- benchmark embeddings from all 15 encoders (about 1 min per run on a T4) ----
UNI = {}                                    # unimodal benchmark probabilities, kept for the supplement
t0 = time.time()
for r, d in RUNS.items():
    for mod in ["C", "D"]:
        for f in range(5):
            dst = f"{OUT}/test_emb/{r}_{mod}_fold{f}.npy"
            if os.path.exists(dst) and os.path.exists(dst.replace(".npy", "_prob.npy")):
                UNI[(r, mod, f)] = np.load(dst.replace(".npy", "_prob.npy")); continue
            net = load_net(d, mod, f)
            P, E = embed_images(net, test[f"path_{mod}"], mod)
            np.save(dst, E.astype(np.float16)); np.save(dst.replace(".npy", "_prob.npy"), P); UNI[(r, mod, f)] = P
            del net; torch.cuda.empty_cache()
    print(f"{r} embedded | {(time.time() - t0) / 60:.1f} min")
for mod in ["C", "D"]:
    Pm = np.mean([v for (r, m, f), v in UNI.items() if m == mod], 0)
    pd.DataFrame(Pm, columns=CLASSES).assign(**{SUB_ID_COL: test["lesion_id"].values})[[SUB_ID_COL] + CLASSES]\
      .to_csv(f"{OUT}/benchmark_unimodal_{mod}_probs.csv", index=False)

In [ ]:
class FusionNet(nn.Module):
    """Late fusion head from notebook 03/08 (kind='late' is the paper model)."""
    def __init__(self, dims, kind="late", d=256, drop=0.3):
        super().__init__()
        self.kind = kind
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Dropout(0.2 if m != "M" else 0.0), nn.Linear(dims[m], d),
                                                    nn.GELU(), nn.LayerNorm(d)) for m in MODS})
        self.heads = nn.ModuleDict({m: nn.Sequential(nn.Dropout(drop), nn.Linear(d, K)) for m in MODS})
    def forward(self, x, avail):                      # avail: (B,3) bool in MODS order
        tok = torch.stack([self.proj[m](x[m]) for m in MODS], 1)
        logits = torch.stack([self.heads[m](tok[:, i]) for i, m in enumerate(MODS)], 1)
        w = avail.float().unsqueeze(-1)
        return (logits * w).sum(1) / w.sum(1).clamp(min=1)

def pattern_mask(p, n):
    return torch.tensor([[m in p for m in MODS]] * n, dtype=torch.bool, device=device)
NONEMPTY = torch.tensor([[m in p for m in MODS] for p in PATTERNS], dtype=torch.bool)
def sample_avail(n, mode, fixed=None):
    if mode == "fixed": return pattern_mask(fixed, n)
    full = torch.ones(n, 3, dtype=torch.bool)
    if mode == "none": return full.to(device)
    sub = NONEMPTY[torch.randint(len(PATTERNS), (n,))]
    keep_full = torch.rand(n) < 0.5
    return torch.where(keep_full.unsqueeze(1), full, sub).to(device)

def evaluate(y, P):
    pred = P.argmax(1)
    out = {"bal_acc": balanced_accuracy_score(y, pred),
           "macro_f1": f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
           "macro_f1_thr05": np.mean([f1_score(y == k, P[:, k] >= 0.5, zero_division=0) for k in range(K)])}
    try: out["macro_auc"] = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: out["macro_auc"] = np.nan
    return out

def to_t(X, idx): return {m: torch.tensor(X[m][idx], device=device) for m in MODS}

@torch.no_grad()
def predict(model, X, idx, pattern):
    model.eval(); x = to_t(X, idx)
    return model(x, pattern_mask(pattern, len(idx))).float().softmax(1).cpu().numpy()

def backbone_val_ids(f):
    """Inner 10% split used for checkpoint selection in notebooks 02/07 (out-of-sample embeddings)."""
    trval = les[les["fold"] != f]
    _, va = train_test_split(trval, test_size=0.10, stratify=trval["y"], random_state=SEED)
    return les.index.isin(va.index)

def train_fusion(X, f, mode="dropout", fixed=None, build=None, epochs=80, patience=15, bs=256, lr=3e-4):
    """Same recipe as notebook 08. Rows 0..len(les)-1 of X are the training lesions; extra rows (benchmark) are never touched."""
    torch.manual_seed(SEED + f); np.random.seed(SEED + f)
    is_val = backbone_val_ids(f)
    tr_idx = np.where((les["fold"].values != f) & ~is_val)[0]
    va_idx = np.where(is_val)[0]
    n_ = np.bincount(y_all[tr_idx], minlength=K).astype(float)
    w = np.sqrt(n_.max() / np.maximum(n_, 1)); w = torch.tensor(w / w.mean(), dtype=torch.float, device=device)
    dims = {m: X[m].shape[1] for m in MODS}
    model = (build(dims) if build else FusionNet(dims, "late")).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    Xtr, ytr = to_t(X, tr_idx), torch.tensor(y_all[tr_idx], device=device)
    best, bad, state = -1, 0, None
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(tr_idx), device=device)
        for s in range(0, len(perm), bs):
            b = perm[s:s + bs]
            xb = {m: Xtr[m][b] for m in MODS}
            loss = F.cross_entropy(model(xb, sample_avail(len(b), mode, fixed)), ytr[b], weight=w, label_smoothing=0.05)
            opt.zero_grad(); loss.backward(); opt.step()
        pats = PATTERNS if mode == "dropout" else [fixed if mode == "fixed" else "CDM"]
        score = np.mean([evaluate(y_all[va_idx], predict(model, X, va_idx, p))["macro_f1"] for p in pats])
        if score > best: best, bad, state = score, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state)
    return model, ep + 1

SITE_CATS = sorted(les["site"].fillna("unknown").unique())
def meta_matrix(df):
    site = pd.Categorical(df["site"].fillna("unknown"), categories=SITE_CATS)
    return np.column_stack([df["age_approx"].fillna(-1).values, df["age_approx"].isna().values,
                            df["skin_tone_class"].values, (df["sex"] == "male").values,
                            pd.get_dummies(site).values]).astype(np.float32)

def standardise(X, tr_mask):
    """Fill missing age with the training median, then z-score every block with TRAIN lesions of this fold only."""
    X = {m: v.copy() for m, v in X.items()}
    age = X["M"][:, 0]; age[age < 0] = np.median(age[tr_mask & (age >= 0)])
    for m in MODS:
        mu, sd = X[m][tr_mask].mean(0), X[m][tr_mask].std(0) + 1e-6
        X[m] = (X[m] - mu) / sd
    return X

In [ ]:
# ---- refit the late_drop fusion head per run and fold, predict OOF (training) and benchmark lesions ----
both = pd.concat([les[["age_approx", "sex", "skin_tone_class", "site"]], test[["age_approx", "sex", "skin_tone_class", "site"]]],
                 ignore_index=True)
META_ALL = meta_matrix(both)
is_tr_row = np.arange(N_TR + N_TE) < N_TR
te_rows = np.arange(N_TR, N_TR + N_TE)
EVAL_PATS = ["CDM", "CM", "CD", "C", "D"]

def fold_inputs(run_dir, r, f, use_monet):
    X = {}
    for m in ["C", "D"]:
        E = np.concatenate([np.load(f"{run_dir}/emb_{m}_fold{f}.npy"), np.load(f"{OUT}/test_emb/{r}_{m}_fold{f}.npy")]).astype(np.float32)
        if use_monet:
            mon = np.concatenate([les[[f"{c}_{m}" for c in MONET]].values, test[[f"{c}_{m}" for c in MONET]].values]).astype(np.float32)
            E = np.concatenate([E, mon], 1)
        X[m] = E
    X["M"] = META_ALL.copy()
    tr_mask = is_tr_row & (np.concatenate([fold, np.full(N_TE, -1)]) != f)
    return standardise(X, tr_mask)

OOF = {p: {} for p in EVAL_PATS}; BENCH = {p: {} for p in EVAL_PATS}; rows = []
t0 = time.time()
for r, d in RUNS.items():
    for p in EVAL_PATS:
        OOF[p][r] = np.zeros((N_TR, K), np.float32); BENCH[p][r] = np.zeros((N_TE, K), np.float32)
    for f in range(5):
        X = fold_inputs(d, r, f, USE_MONET)
        model, n_ep = train_fusion(X, f, mode="dropout")
        te_idx = np.where(fold == f)[0]
        for p in EVAL_PATS:
            OOF[p][r][te_idx] = predict(model, X, te_idx, p)
            BENCH[p][r] += predict(model, X, te_rows, p) / 5
        rows.append(dict(run=r, fold=f, epochs=n_ep, **evaluate(y_all[te_idx], OOF["CDM"][r][te_idx])))
        del model; torch.cuda.empty_cache()
    print(f"{r}: OOF late_drop CDM macro F1 = {evaluate(y_all, OOF['CDM'][r])['macro_f1']:.4f}   "
          f"(should sit within about 0.01 of notebook 08's late_drop_CDM_macro_f1) | {(time.time() - t0) / 60:.1f} min")
pd.DataFrame(rows).to_csv(f"{OUT}/fusion_refit_per_fold.csv", index=False)
np.savez_compressed(f"{OUT}/fused_probs.npz", **{f"oof|{p}|{r}": v for p in EVAL_PATS for r, v in OOF[p].items()},
                    **{f"bench|{p}|{r}": v for p in EVAL_PATS for r, v in BENCH[p].items()})

In [ ]:
# ---- seed ensemble ----
P_OOF = {p: np.mean(list(OOF[p].values()), 0) for p in EVAL_PATS}     # each training lesion: 3 seeds x its own fold model
P_BEN = {p: np.mean(list(BENCH[p].values()), 0) for p in EVAL_PATS}   # each benchmark lesion: 3 seeds x 5 fold models
ens = pd.DataFrame({p: evaluate(y_all, P_OOF[p]) for p in EVAL_PATS}).T
single = pd.DataFrame({p: pd.DataFrame([evaluate(y_all, OOF[p][r]) for r in RUNS]).mean() for p in EVAL_PATS}).T
print("OOF, 3-seed ensemble:"); display(ens.round(4))
print("OOF, mean of single seeds:"); display(single.round(4))

In [ ]:
# ---- per-class thresholds for the ISIC rule (positive iff prob >= 0.5) ----
GRID = np.round(np.arange(0.02, 0.981, 0.01), 2)

def best_thr(p, yk):
    f = [f1_score(yk, p >= t, zero_division=0) for t in GRID]
    return GRID[int(np.argmax(f))]
RARE = np.bincount(y_all, minlength=K) < 20        # MAL_OTH has 9 training lesions: too few to tune a cut-off
def fit_thr(P, yy):
    t = np.array([best_thr(P[:, k], yy == k) for k in range(K)])
    t[RARE] = 0.5                                    # pre-specified: rare classes keep the server's 0.5 rule
    return t

def remap(P, thr, force_top1=False):
    """Monotone per-class map with thr_k -> 0.5, so the server's 0.5 cut reproduces the tuned threshold. AUC is unchanged."""
    Q = np.where(P < thr, 0.5 * P / thr, 0.5 + 0.5 * (P - thr) / (1 - thr))
    if force_top1:                           # every lesion gets at least its top class as positive
        top = P.argmax(1); Q[np.arange(len(Q)), top] = np.maximum(Q[np.arange(len(Q)), top], 0.5001)
    return np.clip(Q, 0, 1)

def isic_f1(yy, Q):                          # the ISIC metric: per-class binary F1 at 0.5, macro average
    return np.mean([f1_score(yy == k, Q[:, k] >= 0.5, zero_division=0) for k in range(K)])

def nested(P, rule):
    """Honest estimate: thresholds fitted on 4 folds, applied to the 5th."""
    Q = np.zeros_like(P)
    for f in range(5):
        tr, te = fold != f, fold == f
        if rule == "raw_0.5": Q[te] = P[te]
        elif rule == "argmax_only": Q[te] = remap(P[te], np.full(K, 0.999), force_top1=True)
        else: Q[te] = remap(P[te], fit_thr(P[tr], y_all[tr]), force_top1=(rule == "thr+top1"))
    return isic_f1(y_all, Q)

RULES = ["raw_0.5", "argmax_only", "thr", "thr+top1"]
choice, est = {}, []
for p in ["CDM", "CM"]:
    sc = {rule: nested(P_OOF[p], rule) for rule in RULES}
    choice[p] = max(sc, key=sc.get)
    est.append(dict(pattern=p, **{f"nested_isicF1_{k}": v for k, v in sc.items()}, chosen=choice[p]))
est = pd.DataFrame(est); est.to_csv(f"{OUT}/threshold_rule_selection.csv", index=False); display(est.round(4))
THR = {p: fit_thr(P_OOF[p], y_all) for p in ["CDM", "CM"]}                    # final thresholds use all 5,240 lesions
display(pd.DataFrame(THR, index=CLASSES).T.round(2))

In [ ]:
# ---- write the submission files ----
def final_probs(p):
    rule = choice[p]
    if rule == "raw_0.5": return P_BEN[p]
    if rule == "argmax_only": return remap(P_BEN[p], np.full(K, 0.999), force_top1=True)
    return remap(P_BEN[p], THR[p], force_top1=(rule == "thr+top1"))

SUBS = {}
for p, name in [("CDM", "submission_full_CDM.csv"), ("CM", "submission_no_dermoscopy_CM.csv")]:
    Q = final_probs(p)
    sub = pd.DataFrame(Q, columns=CLASSES); sub.insert(0, SUB_ID_COL, test["lesion_id"].values)
    assert len(sub) == N_TE and sub[CLASSES].notna().all().all() and sub[CLASSES].values.min() >= 0 and sub[CLASSES].values.max() <= 1
    assert sub[SUB_ID_COL].is_unique
    sub.to_csv(f"{OUT}/{name}", index=False, float_format="%.6f"); SUBS[p] = sub
    pd.DataFrame(P_BEN[p], columns=CLASSES).assign(**{SUB_ID_COL: test["lesion_id"].values})\
      .to_csv(f"{OUT}/benchmark_raw_probs_{p}.csv", index=False)
    print(f"wrote {OUT}/{name}  ({choice[p]})")

# sanity: predicted positives per class on the benchmark vs what training prevalence would predict for 479 lesions
prev = pd.Series(np.bincount(y_all, minlength=K) / N_TR, index=CLASSES)
chk = pd.DataFrame({"expected_from_training": (prev * N_TE).round(0),
                    "pos_CDM": (SUBS["CDM"][CLASSES] >= 0.5).sum(),
                    "pos_CM": (SUBS["CM"][CLASSES] >= 0.5).sum(),
                    "top1_CDM": pd.Series(P_BEN["CDM"].argmax(1)).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES).fillna(0)})
display(chk.astype(int))
for p in ["CDM", "CM"]:
    none_bench = ((SUBS[p][CLASSES] >= 0.5).sum(1) == 0).mean()
    rule = choice[p]
    Q_oof = P_OOF[p] if rule == "raw_0.5" else remap(P_OOF[p], THR[p] if rule.startswith("thr") else np.full(K, 0.999),
                                                      force_top1=rule in ("thr+top1", "argmax_only"))
    none_oof = ((Q_oof >= 0.5).sum(1) == 0).mean()
    print(f"{p}: lesions with no positive class  benchmark {none_bench:.1%} | OOF {none_oof:.1%}")
    if none_bench > max(2 * none_oof, 0.05):
        print(f"   The 5-model average is smoother than the OOF predictions, so fewer lesions cross the thresholds. "
              f"Set choice['{p}'] = 'thr+top1' and rerun this cell.")
SUBS["CDM"].head()

In [ ]:
# ---- summary for the paper (Table: benchmark submission) ----
summary = {
    "runs": ", ".join(RUNS), "encoders": 5 * len(RUNS) * 2, "fusion_heads": 5 * len(RUNS),
    "benchmark_lesions": N_TE, "monet_used": USE_MONET,
    "OOF_macroF1_argmax_CDM_ensemble": ens.loc["CDM", "macro_f1"], "OOF_AUC_CDM_ensemble": ens.loc["CDM", "macro_auc"],
    "OOF_isicF1_raw0.5_CDM": est.set_index("pattern").loc["CDM", "nested_isicF1_raw_0.5"],
    "OOF_isicF1_chosen_rule_CDM": est.set_index("pattern").loc["CDM", f"nested_isicF1_{choice['CDM']}"],
    "OOF_isicF1_chosen_rule_CM": est.set_index("pattern").loc["CM", f"nested_isicF1_{choice['CM']}"],
    "rule_CDM": choice["CDM"], "rule_CM": choice["CM"],
    "preproc_check_min_cosine": float(min(c.min() for c in CHECK.values())),
    "benchmark_score_CDM (fill in from ISIC)": "", "benchmark_score_CM (fill in from ISIC)": ""}
pd.Series(summary).to_csv(f"{OUT}/benchmark_summary.csv"); pd.Series(summary)

## Submitting

1. Download `stage11/submission_full_CDM.csv` from the Output panel.
2. On https://challenge.isic-archive.com/landing/milk10k/ open the submission page, upload the CSV, and add a short approach description.
3. Write the returned macro F1 into `benchmark_score_CDM` in the paper table. If the challenge allows a second submission, upload `submission_no_dermoscopy_CM.csv` too: that gives the hidden-test score for the phone photo + metadata setting, which is the setting the triage claims rest on.
4. If the validator rejects the header, change `SUB_ID_COL` in the first cell and rerun only the last three cells.